# 🧠 02_Algorithm from Scratch: BPE & WordPiece

이 노트북은 `01_theory.md`에서 배운 서브워드 토크나이제이션 알고리즘의 동작 원리를 파이썬 기본 자료구조만을 사용하여 한 단계씩 밑바닥부터 구현해보는 심화 실습입니다.

> **목표**: 라이브러리의 블랙박스를 걷어내고, 수학적 계산과 문자열 병합이 실제로 어떻게 일어나는지 두 눈으로 확인하기!

## 1. BPE (Byte Pair Encoding) 직접 구현하기

이론 문서에 등장한 `low`, `lowest`, `newer`, `wider` 예제를 활용합니다. BPE의 핵심은 **탐욕적 빈도 기반 병합 (Greedy Frequency Merge)** 입니다.

### Step 1. 초기 데이터와 빈도수 정의
단어의 끝을 알 수 있도록 `</w>` 특수 기호를 붙여주고 글자를 띄어쓰기로 분리하여 초기 상태를 만듭니다.

In [1]:
vocab = {
    'l o w </w>': 5,
    'l o w e s t </w>': 2,
    'n e w e r </w>': 6,
    'w i d e r </w>': 3
}

print("==== 초기 상태 ====")
for word, freq in vocab.items():
    print(f"'{word}': {freq}")

==== 초기 상태 ====
'l o w </w>': 5
'l o w e s t </w>': 2
'n e w e r </w>': 6
'w i d e r </w>': 3


### Step 2. 인접 토큰 쌍(Pair) 빈도 계산 함수
`get_stats` 함수는 현재 어휘 사전(vocab)을 순회하며 붙어있는 두 글자/서브워드의 빈도수를 모두 셉니다.

In [2]:
from collections import defaultdict

def get_stats(vocab):
    pairs = defaultdict(int)
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            # 붙어있는 두 기호를 쌍(pair)으로 묶어 빈도를 누적
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

# 현재 상태에서 쌍 빈도수 확인해보기
initial_pairs = get_stats(vocab)
print("초기 쌍 빈도 (일부):", list(initial_pairs.items())[:3])

초기 쌍 빈도 (일부): [(('l', 'o'), 7), (('o', 'w'), 7), (('w', '</w>'), 5)]


### Step 3. 병합(Merge) 함수 구현
가장 빈도가 높은 쌍을 찾아 띄어쓰기를 없애고 하나의 덩어리로 합칩니다. (예: `e` 와 `r` ➔ `er`)

In [3]:
import re

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    # 정규표현식을 이용해 독립적으로 존재하는 두 토큰만 정확히 병합
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

### Step 4. BPE 병합 시뮬레이션 가동 🚀
이제 목표한 횟수(num_merges)만큼 반복하며 실제로 단어들이 어떻게 묶여가는지 확인해봅시다!

In [4]:
num_merges = 3
current_vocab = dict(vocab)

for i in range(num_merges):
    pairs = get_stats(current_vocab)
    if not pairs:
        break
    
    # 가장 빈도가 높은 쌍 찾기
    best_pair = max(pairs, key=pairs.get)
    print(f"\n[{i+1}회차 병합] 🥇 가장 빈도가 높은 쌍: {best_pair} (등장 횟수: {pairs[best_pair]})")
    
    # 병합 수행
    current_vocab = merge_vocab(best_pair, current_vocab)
    
    print("-> 병합 후 어휘 상태:")
    for word, freq in current_vocab.items():
        print(f"   '{word}': {freq}")


[1회차 병합] 🥇 가장 빈도가 높은 쌍: ('e', 'r') (등장 횟수: 9)
-> 병합 후 어휘 상태:
   'l o w </w>': 5
   'l o w e s t </w>': 2
   'n e w er </w>': 6
   'w i d er </w>': 3

[2회차 병합] 🥇 가장 빈도가 높은 쌍: ('er', '</w>') (등장 횟수: 9)
-> 병합 후 어휘 상태:
   'l o w </w>': 5
   'l o w e s t </w>': 2
   'n e w er</w>': 6
   'w i d er</w>': 3

[3회차 병합] 🥇 가장 빈도가 높은 쌍: ('l', 'o') (등장 횟수: 7)
-> 병합 후 어휘 상태:
   'lo w </w>': 5
   'lo w e s t </w>': 2
   'n e w er</w>': 6
   'w i d er</w>': 3


---

## 2. WordPiece Score (PMI) 계산해보기
WordPiece는 단순 빈도가 아니라, 두 토큰을 합쳤을 때 **전체 언어 모델의 확률(우도)이 얼마나 극대화되는가(PMI)**를 평가합니다.

공식: $Score = \frac{P(ab)}{P(a) \times P(b)}$

### Step 1. 가상의 통계량 정의
`a`와 `the`는 단독으로 아주 많이 쓰이지만, `play` 뒤에 `##ing`가 오는 것은 의미적 종속성이 매우 강한 상황을 가정해봅니다.

In [5]:
# 임의의 말뭉치 통계 가정
total_tokens = 10000

# 단일 토큰 등장 횟수
counts = {
    'play': 500,
    '##ing': 800,
    'a': 2000,
    'the': 3000
}

# 결합(연달아 등장) 토큰 등장 횟수
pair_counts = {
    ('play', '##ing'): 450,  # play 뒤에 ing가 매우 자주 옴
    ('a', 'the'): 10         # a 뒤에 the가 연달아 오는 경우는 거의 없음
}

### Step 2. Score 계산 및 증명
공식을 함수로 만들고 계산 결과를 비교합니다.

In [6]:
def calc_wordpiece_score(a, b):
    # 각 확률 계산 (Count / Total)
    p_a = counts[a] / total_tokens
    p_b = counts[b] / total_tokens
    p_ab = pair_counts[(a, b)] / total_tokens
    
    # PMI 기반 Score 계산
    score = p_ab / (p_a * p_b)
    return score, p_a, p_b, p_ab

print("==== WordPiece Score 계산 비교 ====\n")

# 1. 의미적 연결이 강한 경우
score1, p_a1, p_b1, p_ab1 = calc_wordpiece_score('play', '##ing')
print(f"[의미적 종속] 'play' + '##ing'")
print(f" - P(a): {p_a1:.4f}, P(b): {p_b1:.4f}, P(ab): {p_ab1:.4f}")
print(f" - Score: {score1:.2f}\n")

# 2. 의미적 연결이 없는 경우
score2, p_a2, p_b2, p_ab2 = calc_wordpiece_score('a', 'the')
print(f"[단순 독립 단어] 'a' + 'the'")
print(f" - P(a): {p_a2:.4f}, P(b): {p_b2:.4f}, P(ab): {p_ab2:.4f}")
print(f" - Score: {score2:.2f}\n")

==== WordPiece Score 계산 비교 ====

[의미적 종속] 'play' + '##ing'
 - P(a): 0.0500, P(b): 0.0800, P(ab): 0.0450
 - Score: 11.25

[단순 독립 단어] 'a' + 'the'
 - P(a): 0.2000, P(b): 0.3000, P(ab): 0.0010
 - Score: 0.02



> **💡 인사이트 (결론)**
> 단순 빈도로만 치면 `a`와 `the`가 훨씬 많이 등장하지만, WordPiece Score에서는 `play`와 `##ing`의 점수가 **약 22배 이상 압도적으로 높게** 나옵니다. WordPiece는 이를 통해 언어학적인 형태소(어미, 접미사)를 매우 정밀하게 추출해냅니다.